In [2]:
!pip install yfinance


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, mean_absolute_error

In [4]:
# --- Configurable Data Parameters ---
TICKER = "MSFT"  # <--- Change this to TSLA, MSFT, etc., to predict other stocks!
PERIOD = "5y"    # <--- Upgraded to 5 years for deep learning dataset sizes

# Download data
df = yf.download(TICKER, period=PERIOD, interval="1d")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df.reset_index()
df = df.sort_values("Date")
print(f"✅ Downloaded {len(df)} days of historical data for {TICKER}.")


[*********************100%***********************]  1 of 1 completed

✅ Downloaded 1256 days of historical data for MSFT.


In [5]:
# --- Feature Engineering ---

df["sma_10"] = df["Close"].rolling(10).mean()
df["sma_20"] = df["Close"].rolling(20).mean()

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

rolling_mean = df["Close"].rolling(20).mean()
rolling_std  = df["Close"].rolling(20).std()
df["bb_upper"] = rolling_mean + (2 * rolling_std)
df["bb_lower"] = rolling_mean - (2 * rolling_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

df["return"]      = df["Close"].pct_change()
df["return_3d"]   = df["Close"].pct_change(3)
df["return_5d"]   = df["Close"].pct_change(5)

df["volatility_5d"]  = df["return"].rolling(5).std()
df["volatility_10d"] = df["return"].rolling(10).std()

df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]

# Targets
df["target_signal"] = (df["return"].shift(-1) > 0.002).astype(int)
df["target_price"]  = df["Close"].shift(-1)

df = df.dropna()

In [6]:
feature_cols = [
    "rsi_14", "macd", "macd_signal", "bb_width",
    "return_3d", "return_5d", "volatility_5d", "volatility_10d",
    "dist_sma_10", "dist_sma_20", "Volume"
]

X        = df[feature_cols].values
y_signal = df["target_signal"].values
y_price  = df["target_price"].values

split_index = int(len(df) * 0.8)

In [7]:
# Scale features and price target WITHOUT Data Leakage
from sklearn.preprocessing import StandardScaler
import joblib # Added for model persistence later

feat_scaler  = StandardScaler()
price_scaler = StandardScaler()

# 1. Fit ONLY on training data (first 80%)
train_idx = int(len(X) * 0.8)
feat_scaler.fit(X[:train_idx])
price_scaler.fit(y_price[:train_idx].reshape(-1, 1))

# 2. Transform the full dataset
X_scaled       = feat_scaler.transform(X)
y_price_scaled = price_scaler.transform(y_price.reshape(-1, 1)).flatten()

# 3. Build sequences with lookback = 10
LOOKBACK = 10

def create_sequences(X, y_sig, y_prc, lookback):
    Xs, ys_sig, ys_prc = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback:i])
        ys_sig.append(y_sig[i])
        ys_prc.append(y_prc[i])
    return np.array(Xs), np.array(ys_sig), np.array(ys_prc)

X_seq, y_sig_seq, y_prc_seq = create_sequences(X_scaled, y_signal, y_price_scaled, LOOKBACK)

train_size = split_index - LOOKBACK

X_train = torch.tensor(X_seq[:train_size], dtype=torch.float32)
X_test  = torch.tensor(X_seq[train_size:], dtype=torch.float32)

y_sig_train = torch.tensor(y_sig_seq[:train_size], dtype=torch.float32)
y_sig_test  = torch.tensor(y_sig_seq[train_size:], dtype=torch.float32)

y_prc_train = torch.tensor(y_prc_seq[:train_size], dtype=torch.float32)
y_prc_test  = torch.tensor(y_prc_seq[train_size:], dtype=torch.float32)

train_ds = TensorDataset(X_train, y_sig_train, y_prc_train)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=False)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: torch.Size([978, 10, 11]), Test: torch.Size([248, 10, 11])


In [8]:
# --- Dual-output LSTM model ---
class DualLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(hidden_size, 32, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.fc    = nn.Linear(32, 16)
        self.relu  = nn.ReLU()

        # Head 1: signal (BUY/SELL/HOLD)
        self.signal_head = nn.Sequential(nn.Linear(16, 1), nn.Sigmoid())
        # Head 2: next day closing price
        self.price_head  = nn.Linear(16, 1)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.drop1(x)
        x, _ = self.lstm2(x)
        x = self.drop2(x[:, -1, :])   # take last timestep
        x = self.relu(self.fc(x))
        return self.signal_head(x).squeeze(-1), self.price_head(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = DualLSTM(input_size=len(feature_cols)).to(device)
print(model)

DualLSTM(
  (lstm1): LSTM(11, 64, batch_first=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (lstm2): LSTM(64, 32, batch_first=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=32, out_features=16, bias=True)
  (relu): ReLU()
  (signal_head): Sequential(
    (0): Linear(in_features=16, out_features=1, bias=True)
    (1): Sigmoid()
  )
  (price_head): Linear(in_features=16, out_features=1, bias=True)
)


In [9]:
optimizer   = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_bce    = nn.BCELoss()
loss_mse    = nn.MSELoss()

best_val_loss = float("inf")
patience, wait = 10, 0
best_weights   = None

# --- NEW: LOSS WEIGHTING ---
# We force the model to prioritize getting the Trend (Signal) right over the exact Price
ALPHA = 0.8  # 80% weight on Binary Cross Entropy (Signal: Up or Down)
BETA  = 0.2  # 20% weight on Mean Squared Error (Exact Price)

EPOCHS = 100
X_test_dev     = X_test.to(device)
y_sig_test_dev = y_sig_test.to(device)
y_prc_test_dev = y_prc_test.to(device)

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, ysb, ypb in train_dl:
        xb, ysb, ypb = xb.to(device), ysb.to(device), ypb.to(device)
        sig_out, prc_out = model(xb)
        
        # Calculate Weighted Loss mathematically
        loss = (ALPHA * loss_bce(sig_out, ysb)) + (BETA * loss_mse(prc_out, ypb))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Validation loss
    model.eval()
    with torch.no_grad():
        sig_val, prc_val = model(X_test_dev)
        val_loss = (ALPHA * loss_bce(sig_val, y_sig_test_dev) + BETA * loss_mse(prc_val, y_prc_test_dev)).item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | weighted val_loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

# Load the best weights before saving
model.load_state_dict(best_weights)

# --- MODEL PERSISTENCE ---
import os
torch.save(model.state_dict(), 'dual_lstm_model.pth')
joblib.dump(feat_scaler, 'scaler_features.pkl')
joblib.dump(price_scaler, 'scaler_price.pkl')
print("✅ Saved Model & Scalers to disk!")


Epoch  10 | weighted val_loss: 1.3290
Epoch  20 | weighted val_loss: 1.3380
Early stopping at epoch 28
✅ Saved Model & Scalers to disk!


In [10]:
model.eval()
with torch.no_grad():
    sig_prob_t, price_pred_t = model(X_test_dev)

sig_prob    = sig_prob_t.cpu().numpy()
sig_pred    = (sig_prob > 0.5).astype(int)
price_pred  = price_scaler.inverse_transform(price_pred_t.cpu().numpy().reshape(-1, 1)).flatten()
price_actual = price_scaler.inverse_transform(y_prc_test.numpy().reshape(-1, 1)).flatten()

print("LSTM Signal Accuracy:", accuracy_score(y_sig_test.numpy(), sig_pred))
print(classification_report(y_sig_test.numpy(), sig_pred))
print("LSTM ROC-AUC:", roc_auc_score(y_sig_test.numpy(), sig_prob))
print("LSTM Price MAE: $", round(mean_absolute_error(price_actual, price_pred), 2))

LSTM Signal Accuracy: 0.5
              precision    recall  f1-score   support

         0.0       0.53      0.76      0.63       136
         1.0       0.38      0.18      0.24       112

    accuracy                           0.50       248
   macro avg       0.46      0.47      0.44       248
weighted avg       0.46      0.50      0.45       248

LSTM ROC-AUC: 0.4806985294117647
LSTM Price MAE: $ 121.74


In [11]:
# --- Combined output table ---
results = pd.DataFrame({
    "actual_next_close":    price_actual,
    "predicted_next_close": price_pred,
    "actual_signal":        y_sig_test.numpy(),
    "predicted_signal":     sig_pred
})
print(results.tail(10))

     actual_next_close  predicted_next_close  actual_signal  predicted_signal
238         391.790009            318.439209            0.0                 1
239         389.019989            329.298828            0.0                 1
240         381.869995            347.080078            0.0                 0
241         383.000000            375.120605            1.0                 0
242         372.739990            373.618927            0.0                 0
243         371.040009            370.794617            0.0                 0
244         365.970001            365.837769            0.0                 0
245         356.769989            333.534241            0.0                 1
246         358.959991            308.837158            1.0                 1
247         366.250000            298.475067            1.0                 1


In [12]:
# --- Latest signal + next day price forecast ---
prob            = float(sig_prob[-1])
predicted_price = float(price_pred[-1])

if prob > 0.6:
    signal = "BUY"
elif prob < 0.4:
    signal = "SELL"
else:
    signal = "HOLD"

print(f"Signal:               {signal}")
print(f"Confidence:           {round(prob, 2)}")
print(f"Predicted Next Close: ${round(predicted_price, 2)}")

Signal:               HOLD
Confidence:           0.54
Predicted Next Close: $298.48


In [13]:
# --- FASTAPI BACKEND INFERENCE WRAPPER ---
def get_stock_prediction(ticker_symbol: str) -> dict:
    import yfinance as yf
    import pandas as pd
    import numpy as np
    import torch
    import joblib  # <-- Re-added this for the permanent scalers
    
    # 1. Download just enough recent data for lookback + SMA calculations
    raw_df = yf.download(ticker_symbol, period="3mo", interval="1d", progress=False)
    if isinstance(raw_df.columns, pd.MultiIndex):
        raw_df.columns = [col[0] for col in raw_df.columns]
        
    df = raw_df.reset_index()
    if len(df) < 30:
        return {"error": "Not enough data to calculate indicators."}
    
    # 2. Recalculate Indicators (SMA, RSI, MACD, BB, Volatility) on the fly
    df["sma_10"] = df["Close"].rolling(10).mean()
    df["sma_20"] = df["Close"].rolling(20).mean()
    
    delta = df["Close"].diff()
    df["rsi_14"] = 100 - (100 / (1 + (delta.clip(lower=0).rolling(14).mean() / -delta.clip(upper=0).replace(0,0.0001).rolling(14).mean())))
    
    df["macd"] = df["Close"].ewm(span=12, adjust=False).mean() - df["Close"].ewm(span=26, adjust=False).mean()
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    
    df["bb_upper"] = df["Close"].rolling(20).mean() + (2 * df["Close"].rolling(20).std())
    df["bb_lower"] = df["Close"].rolling(20).mean() - (2 * df["Close"].rolling(20).std())
    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]
    
    df["return"]      = df["Close"].pct_change()
    df["return_3d"]   = df["Close"].pct_change(3)
    df["return_5d"]   = df["Close"].pct_change(5)
    df["volatility_5d"]  = df["return"].rolling(5).std()
    df["volatility_10d"] = df["return"].rolling(10).std()
    
    df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
    df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]
    
    df_clean = df.replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(df_clean) < 10: 
        return {"error": "Not enough valid data after calculating features."}
        
    # 3. Form the required 10-day Sequence using your SAVED TRAINING SCALERS
    feature_cols = ["rsi_14", "macd", "macd_signal", "bb_width", "return_3d", "return_5d", "volatility_5d", "volatility_10d", "dist_sma_10", "dist_sma_20", "Volume"]
    recent_chunk = df_clean.tail(10)[feature_cols].values
    
    # <-- FIX: Load the permanent 5-Year Scalers saved by your training cell! -->
    try:
        f_scaler = joblib.load('scaler_features.pkl')
        p_scaler = joblib.load('scaler_price.pkl')
    except Exception:
         return {"error": "Scalers missing! Train the model first."}
         
    recent_scaled = f_scaler.transform(recent_chunk)
    sequence = torch.tensor(recent_scaled, dtype=torch.float32).unsqueeze(0) 
    
    # 4. Load your Trained Model
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        local_model = DualLSTM(input_size=len(feature_cols)).to(device)
        local_model.load_state_dict(torch.load('dual_lstm_model.pth', weights_only=True))
        local_model.eval()
    except Exception:
         return {"error": "Model missing! Train the model first."}
         
    # 5. Execute Live Prediction
    with torch.no_grad():
        sig_prob_t, price_pred_t = local_model(sequence.to(device))
        sig_prob = float(sig_prob_t.cpu().numpy()[0])
        predicted_price = float(p_scaler.inverse_transform(price_pred_t.cpu().numpy().reshape(-1, 1))[0][0])
        
    signal = "BUY" if sig_prob > 0.6 else "SELL" if sig_prob < 0.4 else "HOLD"
    
    return {
        "ticker": ticker_symbol, # Makes sure the ticker name prints accurately
        "signal": signal,
        "confidence_score": round(sig_prob, 3),
        "predicted_next_close": round(predicted_price, 2)
    }

# ==========================================
# THIS ACTS JUST LIKE YOUR API ENDPOINT
# ==========================================
print(get_stock_prediction(TICKER))


{'ticker': 'MSFT', 'signal': 'HOLD', 'confidence_score': 0.552, 'predicted_next_close': 295.5}
